## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [159]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

In [160]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


데이터 불러오기

In [161]:
from datasets import load_dataset

# 2. 데이터셋 로드 (네임스페이스를 명시한 'klue/klue' 경로 사용)
dataset = load_dataset("klue/klue", "ynat")

# 데이터셋 구조 확인
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})


In [162]:
dataset["train"][0]

{'guid': 'ynat-v1_train_00000',
 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영',
 'label': 3,
 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947',
 'date': '2016.06.30. 오전 10:36'}

In [163]:
for i in range(5):
    print("제목:", dataset["train"][i]["title"])
    print("라벨:", dataset["train"][i]["label"])
    print()

제목: 유튜브 내달 2일까지 크리에이터 지원 공간 운영
라벨: 3

제목: 어버이날 맑다가 흐려져…남부지방 옅은 황사
라벨: 3

제목: 내년부터 국가RD 평가 때 논문건수는 반영 않는다
라벨: 2

제목: 김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것
라벨: 2

제목: 회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간
라벨: 3



In [164]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["validation"].shuffle(seed=42).select(range(500))

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['guid', 'title', 'label', 'url', 'date'],
    num_rows: 5000
})
Dataset({
    features: ['guid', 'title', 'label', 'url', 'date'],
    num_rows: 500
})


기본 모델과 토크나이저 불러오기

In [165]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [166]:
model_before = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [167]:
print("모델 레포 이름:", model.config.name_or_path)
print("모델 클래스:", model.__class__.__name__)
print("모델 타입:", model.config.model_type)
print("아키텍처:", model.config.architectures)

모델 레포 이름: klue/bert-base
모델 클래스: BertForSequenceClassification
모델 타입: bert
아키텍처: ['BertForSequenceClassification']


In [168]:
before_classifier = pipeline(
    "text-classification",
    model=model_before,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_texts = [
    "반도체 주식 폭락.",
    "홍명보 사퇴.",
    "여름철 무더위 사망.",
    "트럼프 암살."
]

for text in test_texts:
    result = before_classifier(text)
    print("입력:", text)
    print("파인튜닝 전 결과:", result)
    print()

입력: 반도체 주식 폭락.
파인튜닝 전 결과: [{'label': 'LABEL_2', 'score': 0.22337907552719116}]

입력: 홍명보 사퇴.
파인튜닝 전 결과: [{'label': 'LABEL_2', 'score': 0.22277674078941345}]

입력: 여름철 무더위 사망.
파인튜닝 전 결과: [{'label': 'LABEL_2', 'score': 0.22903572022914886}]

입력: 트럼프 암살.
파인튜닝 전 결과: [{'label': 'LABEL_1', 'score': 0.2213069200515747}]



In [169]:
def tokenize_function(examples):
    return tokenizer(
        examples["title"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [170]:
# 현재 데이터셋에 실제로 어떤 컬럼이 있는지 먼저 확인
print(tokenized_train.column_names)
print(tokenized_test.column_names)

['guid', 'title', 'label', 'url', 'date', 'input_ids', 'token_type_ids', 'attention_mask']
['guid', 'title', 'label', 'url', 'date', 'input_ids', 'token_type_ids', 'attention_mask']


In [171]:
# id 컬럼은 없을 수도 있으므로, 존재하는 컬럼만 제거
remove_cols_train = [col for col in ["guid", "title", "url", "date"] if col in tokenized_train.column_names]
remove_cols_test = [col for col in ["guid", "title", "url", "date"] if col in tokenized_test.column_names]

tokenized_train = tokenized_train.remove_columns(remove_cols_train)
tokenized_test = tokenized_test.remove_columns(remove_cols_test)

# set_format("torch")는 사용하지 않음
tokenized_train[0]

{'label': 3,
 'input_ids': [2,
  29104,
  3956,
  2223,
  2015,
  18556,
  16342,
  15146,
  100,
  1391,
  2437,
  7683,
  8189,
  3,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

In [172]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [173]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [174]:
training_args = TrainingArguments(
    output_dir="./nsmc_klue_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True
)

In [175]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.447020,0.502604,0.846000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model("./my_korean_sentiment_model")
tokenizer.save_pretrained("./my_korean_sentiment_model")

In [ ]:
after_classifier = pipeline(
    "text-classification",
    model="./my_korean_sentiment_model",
    tokenizer="./my_korean_sentiment_model",
    device=0 if torch.cuda.is_available() else -1
)

for text in test_texts:
    result = after_classifier(text)
    print("입력:", text)
    print("파인튜닝 후 결과:", result)
    print()

In [ ]:
# 1. 7개의 카테고리에 맞게 라벨 매핑 수정
label_map = {
    "LABEL_0": "IT과학",
    "LABEL_1": "경제",
    "LABEL_2": "사회",
    "LABEL_3": "생활문화",
    "LABEL_4": "세계",
    "LABEL_5": "스포츠",
    "LABEL_6": "정치"
}

# 함수 이름도 sentiment(감성)에서 topic(주제)으로 바꾸는 게 직관적이겠죠?
def predict_topic(text):
    # after_classifier는 pipeline("text-classification", ...) 객체라고 가정합니다.
    result = after_classifier(text)[0]

    # 모델 출력값(LABEL_0 등)을 한글 카테고리로 변환
    label = label_map[result["label"]]
    score = result["score"]

    print(f"입력 뉴스: {text}")
    print(f"분류 결과: {label}")
    print(f"확신도: {round(score * 100, 2)}%")
    print("-" * 30)

# 2. 뉴스 문장으로 테스트
predict_topic("AI 반도체 기술이 급격히 발전하고 있다.")
predict_topic("코스피 지수가 연일 최고치를 경신했다.")
predict_topic("내일부터 전국적으로 강한 비가 내리겠다.")
predict_topic("이번 주말 개봉하는 영화에 대한 기대감이 높다.")
predict_topic("미국 연준이 금리 인상을 시사했다.")
predict_topic("정부와 여당이 새 예산안에 대해 논의를 시작했다.")
predict_topic("손흥민 선수가 오늘 경기에서 결승골을 터뜨렸다.")

predict_topic("반도체 주식 폭락.")
predict_topic("윤석열 탄핵.")
predict_topic("트럼프 암살.")